# Phase 5 Lab — Reference Solution

**Phase:** Exploratory Data Analysis  
**Scenario:** A retention team needs an exploratory report before approving churn modelling.

**Deliverable:** A reproducible EDA notebook with framing, quality audit, target analysis, segment findings, leakage review, and modelling recommendations.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Define the observation unit, target horizon, and intervention decision.
2. Create a data dictionary and quality audit.
3. Quantify target prevalence and uncertainty.
4. Compare numeric and categorical features by target.
5. Inspect missingness, outliers, interactions, and segments.
6. Create a feature-availability/leakage table.
7. Conclude with limitations and a modelling protocol.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from src.course_utils import dataframe_audit
df=pd.read_csv(DATA_DIR/"customer_churn.csv")
display(dataframe_audit(df))
print("Rows:",len(df),"Churn rate:",df["churn"].mean())

numeric=["tenure_months","monthly_charges","support_tickets_90d","weekly_usage_hours"]
display(df.groupby("churn")[numeric].agg(["mean","median","std"]).round(2))
display(pd.crosstab(df["contract_type"],df["churn"],normalize="index").round(3))
display(pd.crosstab(df["internet_service"],df["churn"],normalize="index").round(3))

availability=pd.DataFrame([
    ["tenure_months","observation cutoff","yes","customer history"],
    ["monthly_charges","observation cutoff","yes","billing system"],
    ["support_tickets_90d","trailing 90 days","yes","support system"],
    ["churn","future outcome window","target only","outcome system"],
],columns=["field","availability","may_be_feature","source"])
display(availability)

fig,ax=plt.subplots(figsize=(7,4))
ax.hist([df.loc[df.churn==0,"tenure_months"],df.loc[df.churn==1,"tenure_months"]],
        bins=24,label=["Retained","Churned"],alpha=.65)
ax.set(title="Tenure distribution by outcome",xlabel="Tenure months",ylabel="Customers")
ax.legend(); plt.show()

print("Recommendation: use stratified splits, pipeline-fitted imputation/encoding, and cost-aware PR/recall metrics.")
print("Limitation: associations are descriptive; intervention effectiveness requires experimental or causal evidence.")

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.